# Quantization Techniques on Different Architectures
## Post-training Quantization and Quantization-aware Training

In [1]:
# Import necessary libraries for file handling, data manipulation, and visualization
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Import libraries for working with images and transformations
from PIL import Image
import cv2 as cv

# Import PyTorch modules for model building, data handling, and evaluation
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torch.nn.functional as F
import torchvision.models as models
import torch.quantization as tq
from torch.quantization import QuantStub, DeQuantStub
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader, Subset
import modelopt.torch.quantization as mtq
import onnxruntime as ort
from torch.ao.quantization import QConfig, default_observer, MinMaxObserver
from torch.ao.quantization import PerChannelMinMaxObserver
from timm import create_model

from torchinfo import summary

# Import libraries for machine learning metrics and model evaluation
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, r2_score, confusion_matrix
# import torchmetric
from tqdm import tqdm
from datetime import datetime
import json
import csv

import warnings
warnings.filterwarnings('ignore')
import gc

# Set the seed.
seed = 42
torch.manual_seed(seed)

/home/sebastian-cruz6/cp-anemia-detection/cawt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_dir="/home/sebastian-cruz6/cp-anemia-detection/data/cp-anemia/"
weights_dir="/home/sebastian-cruz6/cp-anemia-detection/notebooks/weights/"
metrics_dir="/home/sebastian-cruz6/cp-anemia-detection/notebooks/metrics/"

# data_dir = "/content/drive/MyDrive/CAWT_Sebastian_202425/CP-AnemiC/"
# weights_dir = "/content/drive/MyDrive/CAWT_Sebastian_202425/Weights/"
anemic_dir=data_dir+"/Anemic/"
non_anemic_dir=data_dir+"/Non-anemic/"
signature = "QUANTIZATION"

In [3]:
data_sheet_path = data_dir+"Anemia_Data_Collection_Sheet.csv"
data_sheet = pd.read_csv(data_sheet_path)
display(data_sheet)

,IMAGE_ID,HB_LEVEL,Severity,Age(Months),GENDER,REMARK,HOSPITAL,CITY/TOWN,MUNICIPALITY/DISTRICT,REGION,COUNTRY
0,Image_001,9.80,Moderate,6,Female,Anemic,Nkawie-Toase Government Hospital,Nkawie-Toase,Atwima Nwabiagya South,Ashanti,Ghana
1,Image_002,9.90,Moderate,24,Male,Anemic,Ejusu Government Hospital,Ejusu,Ejusu Municipality,Ashanti,Ghana
2,Image_003,11.10,Non-Anemic,24,Female,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
3,Image_004,12.50,Non-Anemic,12,Male,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
4,Image_005,9.90,Moderate,24,Male,Anemic,Sunyani Municipal Hospital,Sunyani,Sunyani Municipality,Bono,Ghana
...,...,...,...,...,...,...,...,...,...,...,...
705,Image_706,12.80,Non-Anemic,48,Male,Non-anemic,Bolgatanga Regional Hospital,Bolgatanga,Bolgatanga Municipality,Upper East,Ghana
706,Image_707,11.47,Non-Anemic,48,Female,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
707,Image_708,11.60,Non-Anemic,60,Male,Non-anemic,Komfo Anokye Teaching Hospital,Kumasi,Kumasi Metropolitan,Ashanti,Ghana
708,Image_709,12.10,Non-Anemic,48,Male,Non-anemic,Bolgatanga Regional Hospital,Bolgatanga,Bolgatanga Municipality,Upper East,Ghana


In [4]:
# Mapping diagnosis to severity
severity_mapping = {
    "Non-Anemic": 0,
    "Mild": 1,
    "Moderate": 2,
    "Severe": 3,
}

data_sheet['Severity'] = data_sheet['Severity'].map(severity_mapping)
display(data_sheet)

,IMAGE_ID,HB_LEVEL,Severity,Age(Months),GENDER,REMARK,HOSPITAL,CITY/TOWN,MUNICIPALITY/DISTRICT,REGION,COUNTRY
0,Image_001,9.80,2,6,Female,Anemic,Nkawie-Toase Government Hospital,Nkawie-Toase,Atwima Nwabiagya South,Ashanti,Ghana
1,Image_002,9.90,2,24,Male,Anemic,Ejusu Government Hospital,Ejusu,Ejusu Municipality,Ashanti,Ghana
2,Image_003,11.10,0,24,Female,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
3,Image_004,12.50,0,12,Male,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
4,Image_005,9.90,2,24,Male,Anemic,Sunyani Municipal Hospital,Sunyani,Sunyani Municipality,Bono,Ghana
...,...,...,...,...,...,...,...,...,...,...,...
705,Image_706,12.80,0,48,Male,Non-anemic,Bolgatanga Regional Hospital,Bolgatanga,Bolgatanga Municipality,Upper East,Ghana
706,Image_707,11.47,0,48,Female,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
707,Image_708,11.60,0,60,Male,Non-anemic,Komfo Anokye Teaching Hospital,Kumasi,Kumasi Metropolitan,Ashanti,Ghana
708,Image_709,12.10,0,48,Male,Non-anemic,Bolgatanga Regional Hospital,Bolgatanga,Bolgatanga Municipality,Upper East,Ghana


In [5]:
# Define data augmentations or transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=np.random.rand()),
    transforms.RandomVerticalFlip(p=np.random.rand()),
    transforms.RandomRotation(degrees=np.random.randint(0, 360)),
    transforms.RandomAffine(degrees=np.random.randint(0, 360)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Custom dataset class
class CPAnemiCDataset(Dataset):
    def __init__(self, dir, df, transform=None):
        self.dir = dir
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['IMAGE_ID']
        img_folder = row['REMARK']
        img_path = os.path.join(self.dir, img_folder, img_id + ".png")
        img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        multiclass_label = torch.tensor(row['Severity'])
        hb_level = torch.tensor(row['HB_LEVEL'])

        return img, multiclass_label, hb_level

    # Load the dataset
image_dataset = CPAnemiCDataset(data_dir, data_sheet, transform=transform)
train_dataset, test_dataset = train_test_split(image_dataset, test_size=0.20, shuffle=True)

print(f"Image Dataset Size (All): {len(image_dataset)}, \
        Train Size: {len(train_dataset)}, \
        Test Size: {len(test_dataset)}")

BATCH_SIZE = 32
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

Image Dataset Size (All): 710,         Train Size: 568,         Test Size: 142


In [6]:
# Default device
device = torch.device('cpu')

# # Check for CUDA availability
# if torch.cuda.is_available():
#     device = torch.device("cuda:1")
# else:
#     print("CUDA is not available, using CPU.")

print(f"Selected device: {device}")

Selected device: cpu


In [7]:
!nvidia-smi

Mon Mar 17 13:54:51 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.120                Driver Version: 550.120        CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:41:00.0 Off |                  Off |
|  0%   50C    P8             46W /  480W |      15MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
class MultiModel(nn.Module):
    MODEL_MAPPING = {
        "mobilenetv2": lambda: models.mobilenet_v2(pretrained=False),
        "resnet18": lambda: models.resnet18(pretrained=False),
        "densenet121": lambda: models.densenet121(pretrained=False),
        "vgg16": lambda: models.vgg16(pretrained=False),
        "vit-tiny": lambda: create_model("vit_tiny_patch16_224", pretrained=False),
        "convnext-tiny": lambda: models.convnext_tiny(pretrained=False),
        "efficientnet-b0": lambda: models.efficientnet_b0(pretrained=False),
        "shufflenetv2-0.5x": lambda: models.shufflenet_v2_x0_5(pretrained=False),
        "regnety-400mf": lambda: models.regnet_y_400mf(pretrained=False),
        "mnasnet0_5": lambda: models.mnasnet0_5(pretrained=False),
        "ghostnetv2": lambda: create_model('ghostnetv2_100.in1k', pretrained=False),
        "tinynet-a": lambda: create_model("tinynet_a.in1k", pretrained=False)
    }

    FEATURE_LAYER_MAPPING = {
        "fc": ["resnet", "shufflenet", "regnet"],
        "classifier": ["densenet", "vgg", "mobilenet", "efficientnet",
                       "mnasnet","convnext", "ghostnet", "tinynet"],
        "head": ["vit"]
    }

    def __init__(self, model_name):
        super().__init__()

        # self.quant = QuantStub() # Start quantization
        
        self.model_name = model_name.lower()

        if self.model_name not in self.MODEL_MAPPING:
            raise ValueError(f"Model {model_name} not supported")

        self.model = self.MODEL_MAPPING[self.model_name]()
        num_ftrs = self._get_feature_size()

        print(f"Initial Backbone {get_model_size(self.model)}")

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.2),
            nn.Linear(num_ftrs, 128),
            nn.ReLU(),
            nn.Linear(128, 5)
        )

        self._assign_classifier()
        # self.dequant = DeQuantStub() # Convert back to fp32
        print(f"Modified Backbone {get_model_size(self.model)}\n")

    def _get_feature_size(self):
        """Retrieve the number of input features for the last layer."""
        # Special case for VGG16 since its features need flattening
        if "vgg" in self.model_name:
            return 25088  # VGG16 outputs (batch, 512, 7, 7) -> flattened to 25088

        feature_layers = {
            "fc": getattr(self.model, "fc", None),
            "classifier": getattr(self.model, "classifier", None),
            "head": getattr(self.model, "head", None)
        }

        for key, layer in feature_layers.items():
            if layer:
                return layer[-1].in_features if isinstance(layer, nn.Sequential) else layer.in_features

        return getattr(self.model, "num_features", None)

    def _assign_classifier(self):
        """Assigns the appropriate classifier to the model based on its architecture."""
        if "vgg" in self.model_name:
            self.model.classifier = self.classifier
        else:
          for attr, models in self.FEATURE_LAYER_MAPPING.items():
            if any(m in self.model_name for m in models):
                setattr(self.model, attr, self.classifier)
                return

    def forward(self, x):
        output = self.model(x)
        return output[:, :4], output[:, 4]  # Class probabilities and Hb level estimate

In [9]:
def get_model_size(model, model_type="pytorch", model_path="model.onnx"):
    """Returns model size in MB"""
    if model_type == "pytorch":
        torch.save(model.state_dict(), "tmp.pt")
        model_size = os.path.getsize("tmp.pt") / 1e6  # Convert bytes to MB
        os.remove("tmp.pt")
    return f"Model Size: {model_size:.2f} MB"

def calibrate_loop(model):
    calibration_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
    for img, _, _ in calibration_loader:
        model(img.to(device))

# Function to measure inference time & memory
def timed_forward(model, img):
    """Measures inference time and memory usage for PyTorch, ONNX, and TensorRT."""
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    # Clear cache
    torch.cuda.empty_cache()
    gc.collect()

    # Record memory usage before inference
    torch.cuda.reset_peak_memory_stats()
    mem_before = torch.cuda.memory_allocated()
    max_mem_before = torch.cuda.max_memory_allocated()

    # Start measuring latency
    start_event.record()

    class_pred, reg_pred = model(img)

    end_event.record()
    torch.cuda.synchronize()  # Ensure accurate timing
    latency = start_event.elapsed_time(end_event)  # Time in ms

    # Record memory usage after inference
    mem_after = torch.cuda.memory_allocated()
    max_mem_after = torch.cuda.max_memory_allocated()

    # Store stats
    stats = {
        "latency": latency,
        "malloc_before": mem_before,
        "malloc_after": mem_after,
        "max_malloc": max_mem_after,
    }

    return class_pred, reg_pred, stats

# Static Weighting Function. Set eta_class to desired importance (Classification > .5, Regression < .5, Equal == .5)
def sw_loss(loss_class, loss_reg, eta_class=0.5):
    eta_reg = 1 - eta_class
    total_loss = (eta_class * loss_class) + (eta_reg * loss_reg)
    return total_loss

In [10]:
models_list = ["mobilenetv2", "resnet18", "densenet121", "vgg16", "vit-tiny",
               "efficientnet-b0", "shufflenetv2-0.5x", "regnety-400mf",
               "mnasnet0_5", "convnext-tiny", "ghostnetv2", "tinynet-a"
               ]
for arch in models_list:
    print(f"Loading model: {arch}")
    model = MultiModel(arch).to(device)
    print(summary(model))
    # print(model)

Loading model: mobilenetv2
Initial Backbone Model Size: 14.24 MB
Modified Backbone Model Size: 9.78 MB

Layer (type:depth-idx)                                  Param #
MultiModel                                              --
├─MobileNetV2: 1-1                                      --
│    └─Sequential: 2-1                                  --
│    │    └─Conv2dNormActivation: 3-1                   928
│    │    └─InvertedResidual: 3-2                       896
│    │    └─InvertedResidual: 3-3                       5,136
│    │    └─InvertedResidual: 3-4                       8,832
│    │    └─InvertedResidual: 3-5                       10,000
│    │    └─InvertedResidual: 3-6                       14,848
│    │    └─InvertedResidual: 3-7                       14,848
│    │    └─InvertedResidual: 3-8                       21,056
│    │    └─InvertedResidual: 3-9                       54,272
│    │    └─InvertedResidual: 3-10                      54,272
│    │    └─InvertedResidual: 3-1

In [11]:
def train(dataloader, model, class_loss, reg1_loss, reg2_loss, optimizer):
    """Trains the model and logs additional metrics."""
    model.train()
    model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')  # Choose backend. FBGEMM is Best for CPUs (like Intel, ARM).
    model = torch.quantization.prepare_qat(model)

    total_loss = 0
    total_ce_loss = 0
    total_mse_loss = 0
    total_mae_loss = 0
    correct = 0
    total_samples = 0

    all_preds = []
    all_targets = []
    all_probs = []
    all_hb_targets = []
    all_hb_preds = []

    for _, (img, multiclass, hb_level) in enumerate(dataloader):
        img = img.to(device)
        multiclass = multiclass.to(device).long()
        hb_level = hb_level.to(device).unsqueeze(1).float()

        optimizer.zero_grad()

        # Forward pass
        class_pred, reg_pred = model(img)

        # Compute losses
        ce_loss = class_loss(class_pred, multiclass)
        mse_loss = reg1_loss(reg_pred, hb_level)
        mae_loss = reg2_loss(reg_pred, hb_level)
        loss = sw_loss(ce_loss, mse_loss, 0.7)  # Weighted loss

        # Backpropagation
        loss.backward()
        optimizer.step()

        # Track total losses
        total_loss += loss.item()
        total_ce_loss += ce_loss.item()
        total_mse_loss += mse_loss.item()
        total_mae_loss += mae_loss.item()

        # Compute classification accuracy
        class_probs = F.softmax(class_pred, dim=1)
        highest_prob_class = torch.argmax(class_probs, dim=1)

        correct += (highest_prob_class == multiclass).sum().item()
        total_samples += multiclass.size(0)

        # Collect data for additional metrics
        all_preds.extend(highest_prob_class.detach().cpu().numpy())
        all_targets.extend(multiclass.detach().cpu().numpy())
        all_probs.extend(class_probs.detach().cpu().numpy())
        all_hb_targets.extend(hb_level.detach().cpu().numpy())
        all_hb_preds.extend(reg_pred.squeeze().cpu().detach().numpy())

    # Compute additional metrics
    precision = precision_score(all_targets, all_preds, average="weighted")
    recall = recall_score(all_targets, all_preds, average="weighted")
    f1 = f1_score(all_targets, all_preds, average="weighted")
    auc = roc_auc_score(all_targets, all_probs, multi_class="ovr")
    r2 = r2_score(all_hb_targets, all_hb_preds)

    # Compute final statistics
    avg_loss = total_loss / len(dataloader)
    avg_ce_loss = total_ce_loss / len(dataloader)
    avg_mse_loss = total_mse_loss / len(dataloader)
    avg_mae_loss = total_mae_loss / len(dataloader)
    accuracy = correct / total_samples

    # Store metrics
    final_metrics = [avg_loss, avg_ce_loss, accuracy, precision, recall, f1, auc, r2, avg_mae_loss, avg_mse_loss]

    return final_metrics


In [12]:
def eval(dataloader, model, class_loss, reg1_loss, reg2_loss, mode=None, precision="fp32"):
    """Evaluates the model with additional metrics: Precision, Recall, AUC, F1, R², Memory Usage, and Latency."""
    model.to(device)
    model.eval()
        
    mean_stats = []
    total_loss = 0
    total_ce_loss = 0
    total_mse_loss = 0
    total_mae_loss = 0
    correct = 0
    total_samples = 0

    all_preds = []
    all_targets = []
    all_probs = []
    all_hb_targets = []
    all_hb_preds = []

    torch.cuda.empty_cache()
    gc.collect()

    if mode == "ptq":
        if precision == "fp16":
            model = model.half()

        if precision == "int8":
            # For Dynamic Quantization
            # model = torch.quantization.quantize_dynamic(
            #      model, {torch.nn.Linear}, dtype=torch.qint8)

            # For Static Quantization
            quant_cfg = mtq.INT8_SMOOTHQUANT_CFG
            mtq.quantize(model, quant_cfg, forward_loop=calibrate_loop)
            mtq.fold_weight(model)

        if precision == "int4":
            quant_cfg = mtq.INT4_AWQ_REAL_QUANT_CFG
            mtq.quantize(model, quant_cfg, forward_loop=calibrate_loop)
            mtq.fold_weight(model)

    with torch.no_grad():
        for _, (img, multiclass, hb_level) in enumerate(dataloader):
            img = img.to(device)
            multiclass = multiclass.to(device).long()
            hb_level = hb_level.to(device).unsqueeze(1).float()
            
            if mode == "ptq":
                if precision == "fp16":
                    img = img.half()
                    hb_level = hb_level.half()

            # Forward pass with latency & memory tracking
            class_pred, reg_pred, stats = timed_forward(model, img)
            mean_stats.append(stats)

            # Compute losses
            ce_loss = class_loss(class_pred, multiclass)
            mse_loss = reg1_loss(reg_pred, hb_level)
            mae_loss = reg2_loss(reg_pred, hb_level)
            loss = sw_loss(ce_loss, mse_loss, 0.7)

            # Track total losses
            total_loss += loss.item()
            total_ce_loss += ce_loss.item()
            total_mse_loss += mse_loss.item()
            total_mae_loss += mae_loss.item()

            # Compute classification accuracy
            class_probs = F.softmax(class_pred, dim=1)
            highest_prob_class = torch.argmax(class_probs, dim=1)

            correct += (highest_prob_class == multiclass).sum().item()
            total_samples += multiclass.size(0)

            # Collect data for additional metrics
            all_preds.extend(highest_prob_class.detach().cpu().numpy())
            all_targets.extend(multiclass.detach().cpu().numpy())
            all_probs.extend(class_probs.detach().cpu().numpy())
            all_hb_targets.extend(hb_level.detach().cpu().numpy())
            all_hb_preds.extend(reg_pred.squeeze().detach().cpu().numpy())

    # Compute mean statistics
    mean_latency = np.mean([s["latency"] for s in mean_stats])
    mean_mem_before = np.mean([s["malloc_before"] for s in mean_stats]) / 1_048_576  # Convert bytes to MB
    mean_mem_after = np.mean([s["malloc_after"] for s in mean_stats]) / 1_048_576  # Convert bytes to MB
    mean_max_mem = np.mean([s["max_malloc"] for s in mean_stats]) / 1_048_576  # Convert bytes to MB

    # Store final mean statistics
    final_mean_stats = [mean_latency, mean_mem_before, mean_mem_after, mean_max_mem]

    # Compute additional evaluation metrics
    prec = precision_score(all_targets, all_preds, average="weighted")
    recall = recall_score(all_targets, all_preds, average="weighted")
    f1 = f1_score(all_targets, all_preds, average="weighted")
    # auc = roc_auc_score(all_targets, all_probs, multi_class="ovr")
    auc = 0.00
    r2 = r2_score(all_hb_targets, all_hb_preds)

    # Compute confusion matrix
    # cm = confusion_matrix(all_targets, all_preds)

    # Compute final average losses
    avg_loss = total_loss / len(dataloader)
    avg_ce_loss = total_ce_loss / len(dataloader)
    avg_mse_loss = total_mse_loss / len(dataloader)
    avg_mae_loss = total_mae_loss / len(dataloader)
    accuracy = correct / total_samples

    # Store metrics
    final_metrics = [avg_loss, avg_ce_loss, accuracy, prec, recall, f1, auc, r2, avg_mae_loss, avg_mse_loss]
    print(get_model_size(model))
    torch.onnx.export(model, img, f"{metrics_dir}/onnx/testing_metrics_{arch}_{signature}_{mode.upper()}_{precision.upper()}.onnx")
    return final_metrics, final_mean_stats

In [13]:
def main(ARCH, BATCH_SIZE=32, EPOCHS=150, FOLDS=5):

    # Define loss functions
    cross_entropy_loss = torch.nn.CrossEntropyLoss()  # Multi-class classification loss
    mse_loss = torch.nn.MSELoss()  # Regression loss
    mae_loss = torch.nn.L1Loss()  # Regression loss

    # Set up 5-Fold Cross Validation
    kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

    print("=" * 100)
    print(f"Training Model: {ARCH}")

    # === INITIALIZE MODEL ===
    model = MultiModel(ARCH).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_val_acc = -float("inf")  # Track best validation accuracy
    train_metrics_list = []
    val_metrics_list = []

    # === TRAINING LOOP ===
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        fold = 1

        for train_idx, val_idx in kf.split(range(len(image_dataset))):
            train_subset = Subset(image_dataset, train_idx)
            val_subset = Subset(image_dataset, val_idx)

            train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
            val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

            if fold == FOLDS:
                # === VALIDATION PHASE ===
                val_metrics, val_stats = eval(val_loader, model, cross_entropy_loss, mse_loss, mae_loss, mode)
                print(
                    f"Validation: Fold {fold} - Total Loss: {val_metrics[0]:.4f}, Cross Entropy: {val_metrics[1]:4f}, Accuracy: {val_metrics[2]:.4f}, "
                    f"Precision: {val_metrics[3]:.4f}, Recall: {val_metrics[4]:.4f}, F1 Score: {val_metrics[5]:.4f}, AUC: {val_metrics[6]:.4f}, "
                    f"R2 Score: {val_metrics[7]:4f}, MAE: {val_metrics[8]:.4f}, MSE: {val_metrics[9]:.4f}"
                )
                print(
                    f"Avg Latency (ms): {val_stats[0]:.2f}, Avg Memory Before (MB): {val_stats[1]:.2f}, "
                    f"Avg Memory After (MB): {val_stats[2]:.2f}, Avg Max Memory (MB): {val_stats[3]:.2f}"
                )

                # Save best model based on validation accuracy
                if val_metrics[2] > best_val_acc:
                    best_val_acc = val_metrics[2]
                    torch.save(
                        model.state_dict(),
                        f"{weights_dir}/pytorch/model_best_accuracy_{ARCH}_{signature}.pth",
                    )
                    print(f"Best model saved with Accuracy: {best_val_acc:.4f}")

                # Store validation metrics
                val_metrics_list.append(
                    {
                        "epoch": epoch + 1,
                        "fold": fold,
                        "total_loss": val_metrics[0],
                        "cross_entropy_loss": val_metrics[1],
                        "accuracy": val_metrics[2],
                        "precision": val_metrics[3],
                        "recall": val_metrics[4],
                        "f1_score": val_metrics[5],
                        "auc": val_metrics[6],
                        "r2_score": val_metrics[7],
                        "mae_loss": val_metrics[8],
                        "mse_loss": val_metrics[9],
                        "latency": val_stats[0],
                        "malloc_before": val_stats[1],
                        "malloc_after": val_stats[2],
                        "max_malloc": val_stats[3],
                    }
                )

            else:
                # === TRAINING PHASE ===
                train_metrics = train(train_loader, model, cross_entropy_loss, mse_loss, mae_loss, optimizer)
                print(
                    f"Training: Fold {fold} - Total Loss: {train_metrics[0]:.4f}, Cross Entropy: {train_metrics[1]:4f}, Accuracy: {train_metrics[2]:.4f}, "
                    f"Precision: {train_metrics[3]:.4f}, Recall: {train_metrics[4]:.4f}, F1 Score: {train_metrics[5]:.4f}, AUC: {train_metrics[6]:.4f}, "
                    f"R2 Score: {train_metrics[7]:4f}, MAE: {train_metrics[8]:.4f}, MSE: {train_metrics[9]:.4f}"
                )

                # Store training metrics
                train_metrics_list.append(
                    {
                        "epoch": epoch + 1,
                        "fold": fold,
                        "total_loss": train_metrics[0],
                        "cross_entropy_loss": train_metrics[1],
                        "accuracy": train_metrics[2],
                        "precision": train_metrics[3],
                        "recall": train_metrics[4],
                        "f1_score": train_metrics[5],
                        "auc": train_metrics[6],
                        "r2_score": train_metrics[7],
                        "mae_loss": train_metrics[8],
                        "mse_loss": train_metrics[9],
                    }
                )

            fold += 1  # Move to next fold
        
        keys = train_metrics_list[0].keys()
        with open(f"{metrics_dir}/pytorch/training_metrics_{ARCH}_{signature}.csv", 'w', newline='') as output_file:
            dict_writer = csv.DictWriter(output_file, keys)
            dict_writer.writeheader()
            dict_writer.writerows(train_metrics_list)

        keys = val_metrics_list[0].keys()
        with open(f"{metrics_dir}/pytorch/validation_metrics_{ARCH}_{signature}.csv", 'w', newline='') as output_file:
            dict_writer = csv.DictWriter(output_file, keys)
            dict_writer.writeheader()
            dict_writer.writerows(val_metrics_list)

    print(f"\nFine-tuned {get_model_size(model)}")
    print("=" * 100)

In [14]:
def infer(arch, path, mode="ptq"):
  
    # Define loss functions
    cross_entropy_loss = torch.nn.CrossEntropyLoss()  # Multi-class classification loss
    mse_loss = torch.nn.MSELoss()  # Regression loss
    mae_loss = torch.nn.L1Loss()  # Regression loss

    test_metrics_list = []
    print("="*100)
    print(f"{arch}")
    
    for bw in ["fp32", "fp16", "int8", "int4"]:
        print(f"\nRunning inference for {bw}")
        print(f"Testing {arch} in {mode} mode")

        model = MultiModel(arch).to(device)
        model.load_state_dict(torch.load(f"{weights_dir}pytorch/model_best_accuracy_{arch}_{path}.pth"))

        # === Testing PHASE ===
        test_metrics, test_stats = eval(test_loader, model, cross_entropy_loss, mse_loss, mae_loss, mode, bw)
        print(
            f"Testing: Total Loss: {test_metrics[0]:.4f}, Cross Entropy: {test_metrics[1]:4f}, Accuracy: {test_metrics[2]:.4f}, "
            f"Precision: {test_metrics[3]:.4f}, Recall: {test_metrics[4]:.4f}, F1 Score: {test_metrics[5]:.4f}, AUC: {test_metrics[6]:.4f}, "
            f"R2 Score: {test_metrics[7]:4f}, MAE: {test_metrics[8]:.4f}, MSE: {test_metrics[9]:.4f}"
        )
        print(
            f"Avg Latency (ms): {test_stats[0]:.2f}, Avg Memory Before (MB): {test_stats[1]:.2f}, "
            f"Avg Memory After (MB): {test_stats[2]:.2f}, Avg Max Memory (MB): {test_stats[3]:.2f}"
        )

        # Store validation metrics
        test_metrics_list.append(
            {
                "model":arch,
                "bit-width": bw,
                "model_size": get_model_size(model),
                "total_loss": test_metrics[0],
                "cross_entropy_loss": test_metrics[1],
                "accuracy": test_metrics[2],
                "precision": test_metrics[3],
                "recall": test_metrics[4],
                "f1_score": test_metrics[5],
                "auc": test_metrics[6],
                "r2_score": test_metrics[7],
                "mae_loss": test_metrics[8],
                "mse_loss": test_metrics[9],
                "latency": test_stats[0],
                "malloc_before": test_stats[1],
                "malloc_after": test_stats[2],
                "max_malloc": test_stats[3],
            }
        )

    keys = test_metrics_list[0].keys()
    with open(f"{metrics_dir}/pytorch/quantization/testing_metrics_{arch}_{signature}_{mode.upper()}.csv", 'w', newline='') as output_file:
        dict_writer = csv.DictWriter(output_file, keys)
        dict_writer.writeheader()
        dict_writer.writerows(test_metrics_list)

## Post-training Quantization

In [15]:
for arch in models_list:
    infer(arch, "BENCHMARK", "ptq")

mobilenetv2

Running inference for fp32
Testing mobilenetv2 in ptq mode
Initial Backbone Model Size: 14.24 MB
Modified Backbone Model Size: 9.78 MB

Model Size: 9.78 MB
Testing: Total Loss: 2.1071, Cross Entropy: 0.424507, Accuracy: 0.7958, Precision: 0.8113, Recall: 0.7958, F1 Score: 0.7935, AUC: 0.0000, R2 Score: -0.034820, MAE: 1.9755, MSE: 6.0331
Avg Latency (ms): 213.73, Avg Memory Before (MB): 0.00, Avg Memory After (MB): 0.00, Avg Max Memory (MB): 0.00

Running inference for fp16
Testing mobilenetv2 in ptq mode
Initial Backbone Model Size: 14.24 MB
Modified Backbone Model Size: 9.78 MB

Model Size: 4.94 MB
Testing: Total Loss: 2.1051, Cross Entropy: 0.422461, Accuracy: 0.7958, Precision: 0.8113, Recall: 0.7958, F1 Score: 0.7935, AUC: 0.0000, R2 Score: -0.034235, MAE: 1.9752, MSE: 6.0305
Avg Latency (ms): 481.35, Avg Memory Before (MB): 0.00, Avg Memory After (MB): 0.00, Avg Max Memory (MB): 0.00

Running inference for int8
Testing mobilenetv2 in ptq mode
Initial Backbone Model S